# <center>🏥 EpiForecast-MX — Avance 3: Modelo Baseline</center>

<center>

**Maestría en Inteligencia Artificial Aplicada — Tecnológico de Monterrey**

TC5035 — Proyecto Integrador · Módulo 3: Ingeniería y Evaluación de Modelos

---

**Pronóstico Epidemiológico de Padecimientos Neurológicos y de Salud Mental en México**

🔴 **Depresión (F32)** · 🟢 **Parkinson (G20)** · 🟡 **Alzheimer (G30)**

---

| Rol | Nombre | Institución |
|---|---|---|
| Estudiante — Desarrollo | Javier Augusto Rebull Saucedo | Tec de Monterrey / Santander US |
| Estudiante — Desarrollo | Juan Carlos Pérez Nava | Tec de Monterrey / IMSS |
| Estudiante — Desarrollo | Luis Gerardo Sánchez Salazar | Tec de Monterrey / Tesla |
| Asesora Académica | Dra. Grettel Barceló Alonso | Tecnológico de Monterrey |
| Sponsor / Líder de Proyecto | Dra. Ruth Pérez | IMSS |
| Investigadora en Psiquiatría | Dra. Lina Díaz Castro | IMSS |

**Equipo 01 · Semana 05 · Febrero 2026**

</center>

---
## Tabla de Contenidos

1. [Configuración del Entorno](#sec1)
2. [Contexto y Justificación del Baseline](#sec2)
3. [Carga y Preparación de Datos](#sec3)
4. [Definición de Métricas de Desempeño](#sec4)
5. [Motor de Modelado y Visualización](#sec5)
6. [🔴 Baseline — Depresión (F32)](#sec6)
   - 6.1 Nacional · 6.2 Nacional × Sexo · 6.3 Por Entidad · 6.4 Entidad × Sexo
7. [🟢 Baseline — Parkinson (G20)](#sec7)
   - 7.1 Nacional · 7.2 Nacional × Sexo · 7.3 Por Entidad · 7.4 Entidad × Sexo
8. [🟡 Baseline — Alzheimer (G30)](#sec8)
   - 8.1 Nacional · 8.2 Nacional × Sexo · 8.3 Por Entidad · 8.4 Entidad × Sexo
9. [📊 Dashboard Ejecutivo Comparativo](#sec9)
10. [Análisis de Sub/Sobreajuste](#sec10)
11. [Análisis de Componentes (Importancia de Características)](#sec11)
12. [Desempeño Mínimo Aceptable](#sec12)
13. [Conclusiones y Siguientes Pasos](#sec13)
14. [Referencias](#sec14)
---

---
## 1. Configuración del Entorno <a id='sec1'></a>

Se importan las bibliotecas necesarias y se establece la configuración visual institucional del IMSS.

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
from pathlib import Path
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

import warnings, sys, logging
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd
import numpy as np
import seaborn as sns
from loguru import logger

# Silenciar logs innecesarios
logging.getLogger("cmdstanpy").disabled = True
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*Disabling.*")
warnings.filterwarnings("ignore", message=".*iteritems.*")

logger.remove()
logger.add(sys.stderr, level="INFO",
           format="{time:HH:mm:ss} | {level:<7} | {message}")
logger.info("Avance 3 — EpiForecast-MX Baseline inicializado")


In [ ]:
# =============================================================================
# CONSTANTES Y CONFIGURACIÓN GLOBAL
# =============================================================================

# --- Rutas -------------------------------------------------------------------
DATA_PATH = Path("../data/processed/data_inegi_General.csv")

# --- Paleta IMSS institucional -----------------------------------------------
IMSS = {
    "black":       "#231F20",
    "burgundy":    "#9B2242",
    "dk_burgundy": "#6F1D46",
    "cool_gray":   "#97999B",
    "teal":        "#00524E",
    "dk_teal":     "#173F35",
    "cream":       "#E8D5B5",
    "gold":        "#B58500",
}

# --- Colores por PADECIMIENTO ------------------------------------------------
PAL = {
    "Depresión": {"c1": "#9B2242", "c2": "#6F1D46", "cl": "#D4758B",
                  "emoji": "🔴", "cie": "F32", "tag": "Depresión (F32)"},
    "Parkinson": {"c1": "#00524E", "c2": "#173F35", "cl": "#4A9E9A",
                  "emoji": "🟢", "cie": "G20", "tag": "Parkinson (G20)"},
    "Alzheimer": {"c1": "#B58500", "c2": "#8B6600", "cl": "#D4B24A",
                  "emoji": "🟡", "cie": "G30", "tag": "Alzheimer (G30)"},
}

# --- Franja COVID-19 ---------------------------------------------------------
COVID_INI = pd.Timestamp("2020-03-15")   # Inicio confinamiento México
COVID_FIN = pd.Timestamp("2021-06-30")   # Fin confinamiento estricto

# --- Parámetros del Baseline -------------------------------------------------
HORIZONTE = 84           # semanas de pronóstico
CV_INITIAL = "730 days"  # ~2 años de entrenamiento inicial
CV_PERIOD  = "56 days"   # ~8 semanas entre cortes
CV_HORIZON = "168 days"  # ~24 semanas de horizonte de evaluación

# --- Estilo global matplotlib ------------------------------------------------
plt.rcParams.update({
    "figure.facecolor":   "#FAFAFA",
    "axes.facecolor":     "#FAFAFA",
    "axes.edgecolor":     IMSS["cool_gray"],
    "axes.labelcolor":    IMSS["black"],
    "text.color":         IMSS["black"],
    "xtick.color":        IMSS["black"],
    "ytick.color":        IMSS["black"],
    "axes.grid":          True,
    "grid.alpha":         0.20,
    "grid.color":         IMSS["cool_gray"],
    "font.family":        "sans-serif",
    "font.size":          11,
    "axes.titlesize":     13,
    "axes.titleweight":   "bold",
    "figure.titlesize":   15,
    "figure.titleweight": "bold",
    "figure.dpi":         140,
    "savefig.dpi":        160,
    "savefig.bbox":       "tight",
})

logger.success(f"Config cargada | 3 padecimientos × 4 niveles | Horizonte: {HORIZONTE} sem")


---
## 2. Contexto y Justificación del Baseline <a id='sec2'></a>

### 2.1 Objetivo

El propósito de este avance es construir un **modelo de referencia (baseline)** para **cada uno de los tres padecimientos** — Depresión (F32), Parkinson (G20) y Alzheimer (G30) — que permita:

1. **Evaluar la viabilidad** del pronóstico epidemiológico automatizado para cada padecimiento.
2. **Establecer un piso de desempeño** contra el cual se compararán modelos optimizados en Avance 4.
3. **Gestionar expectativas** con los stakeholders del IMSS (Dra. Ruth, Dra. Lina).
4. **Validar el pipeline** de datos SINAVE–INEGI para las tres enfermedades.

### 2.2 ¿Por qué Prophet como Algoritmo Baseline?

La selección de **Prophet** (Taylor & Letham, 2018) se fundamenta en los criterios CRISP-ML(Q) (Studer et al., 2021):

| Criterio CRISP-ML(Q) | Justificación |
|---|---|
| **Tipo de datos** | Series de tiempo semanales con estacionalidad anual y tendencia. Prophet está diseñado específicamente para este tipo de patrón. |
| **Robustez** | Maneja valores faltantes, outliers y *changepoints* de forma nativa — crítico dado el efecto disruptivo del COVID-19 en los tres padecimientos. |
| **Escalabilidad** | Pipeline parametrizable: ejecuta modelos idénticos por padecimiento, entidad y sexo sin cambio de arquitectura. |
| **Interpretabilidad** | Descompone la serie en tendencia + estacionalidad, facilitando la comunicación con médicos y tomadores de decisiones del IMSS. |
| **Intervalos de predicción** | Genera bandas de incertidumbre nativamente — requisito fundamental para planificación en salud pública. |
| **Evidencia previa** | La Fase I demostró que modelos ML recursivos (XGBoost, Random Forest) presentan acumulación progresiva de error en horizontes largos (52+ semanas). |

### 2.3 Descarte de Modelos Alternativos

| Modelo | Razón de descarte |
|---|---|
| **XGBoost / RF recursivo** | La Dra. Grettel identificó *data leakage* en la implementación de la Fase I (reunión 12-feb-2026). Adicionalmente, la predicción recursiva multi-paso acumula error en horizontes de 84 semanas. |
| **ARIMA / SARIMA** | Requieren estacionariedad estricta y especificación manual de órdenes. No manejan los *changepoints* por COVID-19 de forma nativa. |
| **Modelo naïve** | No captura la estacionalidad anual documentada en los tres padecimientos neurológicos y de salud mental. |

### 2.4 Estructura de Modelado por Padecimiento

Para **cada uno de los tres padecimientos** se construyen modelos en **cuatro niveles de granularidad**:

| Nivel | Modelos por padecimiento | Propósito |
|---|---|---|
| **Nacional** | 1 | Visión macro del comportamiento epidemiológico |
| **Nacional × Sexo** | 2 | Identificar diferencias de género en la incidencia |
| **Por Entidad** | 32 | Pronóstico estatal para asignación de recursos |
| **Entidad × Sexo** | 64 | Máxima granularidad para planificación operativa |

**Total: 3 padecimientos × 99 modelos = 297 modelos baseline.**

---
## 3. Carga y Preparación de Datos <a id='sec3'></a>

Se carga el dataset consolidado SINAVE–INEGI y se segmentan las series temporales **por padecimiento** para alimentar el motor de modelado.

In [ ]:
# =============================================================================
# CARGA DE DATOS
# =============================================================================
df_raw = pd.read_csv(DATA_PATH)
df_raw["Fecha"] = pd.to_datetime(df_raw["Fecha"])

logger.info(f"Dataset: {df_raw.shape[0]:,} registros × {df_raw.shape[1]} columnas")
logger.info(f"Rango: {df_raw['Fecha'].min():%Y-%m-%d} → {df_raw['Fecha'].max():%Y-%m-%d}")
print(f"Columnas: {list(df_raw.columns)}")
df_raw.head()


In [ ]:
# =============================================================================
# DETECCIÓN AUTOMÁTICA DE COLUMNA DE PADECIMIENTO
# =============================================================================
# Buscar la columna que identifica cada enfermedad
candidatas = ["Padecimiento", "padecimiento", "Enfermedad", "enfermedad",
              "Diagnostico", "diagnostico", "CIE", "Clave"]

COL_PAD = None
for c in candidatas:
    if c in df_raw.columns:
        COL_PAD = c
        break

# Si no hay columna explícita, buscar en columnas categóricas
if COL_PAD is None:
    for c in df_raw.select_dtypes(include="object").columns:
        vals = df_raw[c].unique()
        # Buscar si alguna columna tiene valores que contienen F32/G20/G30
        txt = " ".join(str(v) for v in vals).lower()
        if any(k in txt for k in ["depres", "parkinson", "alzheim", "f32", "g20", "g30"]):
            COL_PAD = c
            break

if COL_PAD:
    logger.success(f"Columna de padecimiento: '{COL_PAD}'")
    print(f"Valores únicos: {df_raw[COL_PAD].unique()}")
else:
    logger.warning("⚠ No se encontró columna de padecimiento explícita.")
    print("Columnas disponibles:", list(df_raw.columns))
    print("\nSe intentará inferir la estructura...")


In [ ]:
# =============================================================================
# PREPARACIÓN DE SERIES POR PADECIMIENTO
# =============================================================================

def preparar_series(df, filtro_pad=None, col_pad=None):
    """
    Prepara las series temporales a los 4 niveles de granularidad
    para un padecimiento específico.
    """
    # Filtrar por padecimiento si aplica
    if filtro_pad and col_pad:
        mask = df[col_pad].str.contains(filtro_pad, case=False, na=False)
        d = df[mask].copy()
    else:
        d = df.copy()

    if len(d) == 0:
        return None

    # --- Nacional (suma total) ---
    nac = (d.groupby("Fecha")[["incrementos_hombres", "incrementos_mujeres"]]
            .sum().reset_index().rename(columns={"Fecha": "ds"}))
    nac["y"] = nac["incrementos_hombres"] + nac["incrementos_mujeres"]
    nac = nac.sort_values("ds").reset_index(drop=True)

    # --- Nacional Hombres / Mujeres ---
    nh = (d.groupby("Fecha")["incrementos_hombres"].sum().reset_index()
           .rename(columns={"Fecha": "ds", "incrementos_hombres": "y"})
           .sort_values("ds").reset_index(drop=True))
    nm = (d.groupby("Fecha")["incrementos_mujeres"].sum().reset_index()
           .rename(columns={"Fecha": "ds", "incrementos_mujeres": "y"})
           .sort_values("ds").reset_index(drop=True))

    # --- Por Estado ---
    edo = (d.groupby(["Fecha", "Entidad"])[["incrementos_hombres", "incrementos_mujeres"]]
            .sum().reset_index().rename(columns={"Fecha": "ds"}))
    edo["y"] = edo["incrementos_hombres"] + edo["incrementos_mujeres"]

    # --- Estado × Sexo ---
    eh = (d.groupby(["Fecha", "Entidad"])["incrementos_hombres"].sum().reset_index()
           .rename(columns={"Fecha": "ds", "incrementos_hombres": "y"}))
    em = (d.groupby(["Fecha", "Entidad"])["incrementos_mujeres"].sum().reset_index()
           .rename(columns={"Fecha": "ds", "incrementos_mujeres": "y"}))

    entidades = sorted(edo["Entidad"].unique())

    return {"nac": nac, "nac_h": nh, "nac_m": nm,
            "edo": edo, "edo_h": eh, "edo_m": em,
            "entidades": entidades}


# --- Extraer series por cada padecimiento ---
DATOS = {}
for nombre in ["Depresión", "Parkinson", "Alzheimer"]:
    logger.info(f"Preparando series → {nombre}")
    DATOS[nombre] = preparar_series(df_raw, nombre, COL_PAD)
    if DATOS[nombre]:
        n = len(DATOS[nombre]["nac"])
        ne = len(DATOS[nombre]["entidades"])
        logger.success(f"  {nombre}: {n} semanas × {ne} entidades")
    else:
        logger.warning(f"  {nombre}: sin datos separados")

# Fallback: si no hay separación, usar todo como padecimiento único
if all(v is None for v in DATOS.values()):
    logger.warning("No se pudo separar por padecimiento. Usando dataset completo.")
    DATOS["General"] = preparar_series(df_raw)
    # Actualizar PAL para usar General
    PAL["General"] = PAL["Depresión"].copy()
    PAL["General"]["tag"] = "General (3 padecimientos combinados)"

logger.success("Datos preparados por padecimiento")


---
## 4. Definición de Métricas de Desempeño <a id='sec4'></a>

### 4.1 Selección y Justificación

Para un problema de **regresión de series de tiempo** epidemiológicas, las métricas se eligen considerando la necesidad de comparar entre padecimientos con volúmenes de incidencia muy distintos (Depresión >> Parkinson > Alzheimer) y la interpretabilidad para stakeholders no técnicos.

| Métrica | Fórmula | Rol en el proyecto |
|---|---|---|
| **MAPE** ⭐ | $\frac{100}{n}\sum\frac{|y_i - \hat{y}_i|}{|y_i|}$ | **Métrica principal**: comparabilidad entre padecimientos y estados. Interpretable: "error promedio en %". |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Penaliza errores extremos que afectan la asignación de recursos. |
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Directamente interpretable como "diferencia promedio en casos". |
| **MDAPE** | $\text{mediana}\left(\frac{|y_i - \hat{y}_i|}{|y_i|}\right)$ | Robusta ante outliers. Complementa al MAPE. |

Se excluyen observaciones con $y_i = 0$ del cálculo de MAPE/MDAPE (estados con bajo reporte, advertido por la Dra. Grettel en reunión del 12-feb).

### 4.2 Cross-Validation Temporal

Siguiendo la recomendación de la Dra. Grettel:

- **initial**: 730 días (~2 años de entrenamiento mínimo)
- **period**: 56 días (~8 semanas entre cortes de validación)
- **horizon**: 168 días (~24 semanas de horizonte evaluado)

---
## 5. Motor de Modelado y Visualización <a id='sec5'></a>

Funciones reutilizables con estilo institucional IMSS. Cada gráfico incluye:

- 🟢 **Puntos verdes**: observaciones reales (incrementos semanales)
- 🔴 **Línea roja**: pronóstico Prophet ($\hat{y}$)
- 🔵 **Banda sombreada**: intervalo de predicción al 80%
- 🟥 **Franja roja transparente**: período COVID-19 (mar 2020 – jun 2021)
- 🔺 **Triángulos rojos**: valores atípicos detectados por IQR

In [ ]:
# =============================================================================
# FUNCIONES DE MÉTRICAS Y DETECCIÓN DE OUTLIERS
# =============================================================================

def calc_mape(yt, yp):
    """MAPE excluyendo y_true == 0."""
    d = yt.replace(0, np.nan)
    return (np.abs((yt - yp) / d) * 100).mean()

def calc_rmse(yt, yp):
    return np.sqrt(np.mean((yt - yp) ** 2))

def calc_mae(yt, yp):
    return np.mean(np.abs(yt - yp))

def detectar_outliers(y, factor=1.5):
    """Método IQR para detección de outliers."""
    Q1, Q3 = y.quantile(0.25), y.quantile(0.75)
    IQR = Q3 - Q1
    return (y < Q1 - factor * IQR) | (y > Q3 + factor * IQR)

logger.success("Funciones de métricas definidas")


In [ ]:
# =============================================================================
# VISUALIZACIÓN PROFESIONAL — PRONÓSTICO CON LEYENDA CLARA
# =============================================================================

def graficar_pronostico(modelo, forecast, serie, titulo, pal,
                        mostrar_componentes=True):
    """
    Gráfico de pronóstico profesional con leyenda explícita.

    LEYENDA:
    - 🟢 Puntos: observaciones reales (incrementos semanales)
    - 🔴 Línea: pronóstico Prophet (ŷ)
    - Banda sombreada: intervalo de predicción (80%)
    - Franja roja: período COVID-19
    - △ Triángulos: outliers detectados (IQR)
    """
    out_mask = detectar_outliers(serie["y"])
    outliers = serie[out_mask]
    fecha_max = serie["ds"].max()
    y_max = serie["y"].max()

    fig, ax = plt.subplots(figsize=(17, 5.5))

    # ── 1. FRANJA COVID-19 ──────────────────────────────────────────────────
    ax.axvspan(COVID_INI, COVID_FIN, alpha=0.10, color="#E53935", zorder=0)
    mid = COVID_INI + (COVID_FIN - COVID_INI) / 2
    ax.annotate("COVID-19", xy=(mid, y_max * 0.96),
                fontsize=9, fontweight="bold", color="#C62828", ha="center",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#C62828",
                          alpha=0.85, lw=1.2))

    # ── 2. INTERVALO DE PREDICCIÓN (banda) ──────────────────────────────────
    ax.fill_between(forecast["ds"], forecast["yhat_lower"], forecast["yhat_upper"],
                    alpha=0.18, color=pal["c1"], zorder=1,
                    label="Intervalo de predicción (80%)")

    # ── 3. OBSERVACIONES REALES (puntos verdes) ────────────────────────────
    ax.scatter(serie["ds"], serie["y"],
               s=10, color=IMSS["teal"], alpha=0.50, zorder=3,
               label="● Observaciones reales (incrementos semanales)")

    # ── 4. LÍNEA DE PRONÓSTICO PROPHET (línea roja) ────────────────────────
    ax.plot(forecast["ds"], forecast["yhat"],
            color=IMSS["burgundy"], linewidth=1.6, zorder=4,
            label="── Pronóstico Prophet (ŷ)")

    # ── 5. OUTLIERS (triángulos) ───────────────────────────────────────────
    if len(outliers) > 0:
        ax.scatter(outliers["ds"], outliers["y"],
                   marker="^", s=55, color="#FF1744", edgecolors="#B71C1C",
                   linewidths=0.8, zorder=5,
                   label=f"▲ Outliers detectados (IQR) — n={len(outliers)}")

    # ── 6. DIVISOR DATOS / PRONÓSTICO ──────────────────────────────────────
    ax.axvline(fecha_max, color=IMSS["cool_gray"], ls=":", lw=1.2, alpha=0.7)
    ax.text(fecha_max, y_max * 0.03, "  ← Datos │ Pronóstico →",
            fontsize=8, color=IMSS["cool_gray"], va="bottom")

    # ── Formato ────────────────────────────────────────────────────────────
    ax.set_title(titulo, fontsize=14, fontweight="bold", pad=12)
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Incrementos Semanales")
    ax.legend(loc="upper left", fontsize=8.5, framealpha=0.92,
              fancybox=True, borderpad=0.8, handletextpad=0.6)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    # ── COMPONENTES ────────────────────────────────────────────────────────
    if mostrar_componentes:
        fig2 = modelo.plot_components(forecast)
        fig2.suptitle(f"Descomposición — {titulo}", fontsize=12,
                      fontweight="bold", y=1.02)
        for a in fig2.get_axes():
            a.set_facecolor("#FAFAFA")
            for ln in a.get_lines():
                ln.set_color(pal["c1"]); ln.set_linewidth(2)
            for col in a.collections:
                col.set_facecolor(pal["cl"]); col.set_alpha(0.25)
        plt.tight_layout(); plt.show(); plt.close(fig2)

    return len(outliers)


def graficar_cv_horizonte(df_pm, titulo, color):
    """Degradación del error por horizonte de pronóstico."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
    fig.suptitle(f"Degradación del Error — {titulo}", fontsize=12,
                 fontweight="bold")
    for ax, (m, lab) in zip(axes,
            [("rmse","RMSE"), ("mae","MAE"), ("mdape","MDAPE")]):
        if m not in df_pm.columns:
            continue
        dias = df_pm["horizon"].dt.days
        ax.plot(dias, df_pm[m], color=color, lw=2.2)
        ax.fill_between(dias, 0, df_pm[m], alpha=0.08, color=color)
        ax.set_xlabel("Horizonte (días)"); ax.set_ylabel(lab)
        ax.set_title(lab, fontweight="bold")
    plt.tight_layout(); plt.show(); plt.close(fig)


logger.success("Motor de visualización profesional definido")


In [ ]:
# =============================================================================
# MOTOR PRINCIPAL DE EVALUACIÓN BASELINE
# =============================================================================

def evaluar_baseline(serie, nombre, pal, graficar=True, componentes=True,
                     verbose=True):
    """
    Entrena Prophet baseline → cross-validation → métricas → visualización.
    Retorna dict con modelo, forecast, métricas y resumen.
    """
    m = Prophet()
    m.fit(serie)
    fut = m.make_future_dataframe(periods=HORIZONTE, freq="W")
    fc  = m.predict(fut)

    cv  = cross_validation(m, initial=CV_INITIAL, period=CV_PERIOD,
                           horizon=CV_HORIZON)
    pm  = performance_metrics(cv)

    # Resumen de métricas CV
    cols = [c for c in ["rmse", "mae", "mdape"] if c in pm.columns]
    r = pm[cols].mean(numeric_only=True).to_dict()
    r["mape"] = calc_mape(cv["y"], cv["yhat"])
    r["nombre"] = nombre

    # Métricas en entrenamiento
    merged = serie.merge(fc[fc["ds"].isin(serie["ds"])][["ds","yhat"]], on="ds")
    r["rmse_train"] = calc_rmse(merged["y"], merged["yhat"])
    r["mae_train"]  = calc_mae(merged["y"],  merged["yhat"])
    r["mape_train"] = calc_mape(merged["y"], merged["yhat"])

    if verbose:
        logger.info(f"  {nombre}")
        logger.info(f"    CV  → RMSE:{r.get('rmse',0):.1f} MAE:{r.get('mae',0):.1f} "
                     f"MAPE:{r['mape']:.1f}% MDAPE:{r.get('mdape',0):.4f}")
        logger.info(f"    Train → MAPE:{r['mape_train']:.1f}%")

    n_out = 0
    if graficar:
        n_out = graficar_pronostico(m, fc, serie, nombre, pal,
                                    mostrar_componentes=componentes)
        graficar_cv_horizonte(pm, nombre, pal["c1"])
    r["n_outliers"] = n_out

    return {"modelo": m, "forecast": fc, "cv": cv, "pm": pm, "r": r}


def ejecutar_estados(df_edo, entidades, pad_tag, pal,
                     graficos_en=None):
    """Entrena modelos para todas las entidades."""
    if graficos_en is None:
        graficos_en = ["Ciudad de México", "Jalisco", "Nuevo León", "Oaxaca"]
    res = []
    for i, e in enumerate(entidades, 1):
        s = df_edo.loc[df_edo["Entidad"] == e, ["ds","y"]].copy()
        ver = e in graficos_en
        try:
            r = evaluar_baseline(s, f"{pad_tag} — {e}", pal,
                                 graficar=ver, componentes=False, verbose=ver)
            res.append(r["r"])
        except Exception as ex:
            if ver: logger.error(f"  Error {e}: {ex}")
            res.append({"nombre": f"{pad_tag}—{e}", "mape": np.nan,
                         "rmse": np.nan, "mae": np.nan, "mdape": np.nan,
                         "mape_train": np.nan, "rmse_train": np.nan,
                         "mae_train": np.nan, "n_outliers": 0})
        if i % 8 == 0:
            logger.info(f"    Progreso: {i}/{len(entidades)}")
    return res


def ejecutar_estado_sexo(df_h, df_m, entidades, pad_tag, pal):
    """Modelos Estado×Sexo (solo métricas, sin gráficos)."""
    res = []
    for e in entidades:
        for sexo, df in [("Hombres", df_h), ("Mujeres", df_m)]:
            s = df.loc[df["Entidad"] == e, ["ds","y"]].copy()
            try:
                r = evaluar_baseline(s, f"{pad_tag}—{e}—{sexo}", pal,
                                     graficar=False, verbose=False)
                r["r"]["estado"] = e; r["r"]["sexo"] = sexo
                res.append(r["r"])
            except:
                res.append({"nombre": f"{pad_tag}—{e}—{sexo}",
                             "estado": e, "sexo": sexo,
                             "mape": np.nan, "rmse": np.nan, "mae": np.nan,
                             "mape_train": np.nan, "n_outliers": 0})
    return res


logger.success("Motor de evaluación baseline completo")


In [ ]:
# =============================================================================
# EJECUTOR MAESTRO POR PADECIMIENTO
# =============================================================================

def ejecutar_padecimiento(nombre, datos, pal):
    """
    Ejecuta los 4 niveles de baseline para un padecimiento completo.
    Retorna diccionario con todos los resultados.
    """
    tag = pal["tag"]
    e = pal["emoji"]

    print("\n")
    print("╔" + "═" * 78 + "╗")
    print(f"║  {e}  BASELINE — {tag.upper():<67}║")
    print("╚" + "═" * 78 + "╝")

    R = {}

    # ── NIVEL 1: NACIONAL ────────────────────────────────────────────────
    print(f"\n{'─'*65}\n  {e} Nivel 1: NACIONAL\n{'─'*65}")
    R["nac"] = evaluar_baseline(datos["nac"][["ds","y"]].copy(),
                                f"{tag} — Nacional", pal,
                                graficar=True, componentes=True)

    # ── NIVEL 2: NACIONAL × SEXO ────────────────────────────────────────
    print(f"\n{'─'*65}\n  {e} Nivel 2: NACIONAL × SEXO\n{'─'*65}")
    pal_h = {"c1": IMSS["teal"],    "cl": "#4A9E9A", "c2": IMSS["dk_teal"]}
    pal_m = {"c1": IMSS["burgundy"],"cl": "#D4758B", "c2": IMSS["dk_burgundy"]}
    R["nac_h"] = evaluar_baseline(datos["nac_h"][["ds","y"]].copy(),
                                  f"{tag} — Nacional Hombres", pal_h,
                                  graficar=True, componentes=False)
    R["nac_m"] = evaluar_baseline(datos["nac_m"][["ds","y"]].copy(),
                                  f"{tag} — Nacional Mujeres", pal_m,
                                  graficar=True, componentes=False)

    # ── NIVEL 3: POR ENTIDAD ────────────────────────────────────────────
    n_edos = len(datos["entidades"])
    print(f"\n{'─'*65}\n  {e} Nivel 3: POR ENTIDAD ({n_edos} estados)\n{'─'*65}")
    R["edos"] = ejecutar_estados(datos["edo"], datos["entidades"], tag, pal)

    # ── NIVEL 4: ENTIDAD × SEXO ────────────────────────────────────────
    n_es = n_edos * 2
    print(f"\n{'─'*65}\n  {e} Nivel 4: ENTIDAD × SEXO ({n_es} modelos)\n{'─'*65}")
    R["edo_sexo"] = ejecutar_estado_sexo(datos["edo_h"], datos["edo_m"],
                                         datos["entidades"], tag, pal)

    total = 1 + 2 + len(R["edos"]) + len(R["edo_sexo"])
    logger.success(f"  {e} {tag}: {total} modelos completados")

    return R

logger.success("Ejecutor maestro definido")


---
## 6. 🔴 Baseline — Depresión (F32) <a id='sec6'></a>

La **Depresión (CIE-10: F32)** es el padecimiento con **mayor volumen de incidencia** reportada en SINAVE. Se espera que el modelo capture la estacionalidad anual bien definida y el efecto disruptivo del COVID-19 en las consultas de salud mental.

> *"La Depresión muestra un incremento sostenido post-pandemia, probablemente asociado a mayor conciencia y detección."* — Dra. Lina Díaz Castro, IMSS (reunión 11-feb-2026)

In [ ]:
# =============================================================================
# 🔴 EJECUTAR BASELINE COMPLETO — DEPRESIÓN (F32)
# =============================================================================
if DATOS.get("Depresión") is not None:
    RES_DEP = ejecutar_padecimiento("Depresión", DATOS["Depresión"],
                                     PAL["Depresión"])
elif "General" in DATOS:
    logger.warning("Usando datos generales como proxy para Depresión")
    RES_DEP = ejecutar_padecimiento("General", DATOS["General"],
                                     PAL.get("General", PAL["Depresión"]))
else:
    RES_DEP = None
    logger.error("No hay datos disponibles para Depresión")


---
## 7. 🟢 Baseline — Parkinson (G20) <a id='sec7'></a>

La **Enfermedad de Parkinson (CIE-10: G20)** presenta un volumen de incidencia significativamente menor que la Depresión. La Dra. Ruth indicó que Parkinson muestra una **continuación de la tendencia de crecimiento** previa a la pandemia, sin alteraciones visibles en el patrón post-COVID.

> *"Parkinson tiene un comportamiento más estable que Depresión y Alzheimer; la tendencia creciente se asocia al envejecimiento poblacional."* — Dra. Ruth Pérez, IMSS (reunión 11-feb-2026)

In [ ]:
# =============================================================================
# 🟢 EJECUTAR BASELINE COMPLETO — PARKINSON (G20)
# =============================================================================
if DATOS.get("Parkinson") is not None:
    RES_PAR = ejecutar_padecimiento("Parkinson", DATOS["Parkinson"],
                                     PAL["Parkinson"])
else:
    RES_PAR = None
    logger.info("Parkinson: datos no disponibles por separado")


---
## 8. 🟡 Baseline — Alzheimer (G30) <a id='sec8'></a>

La **Enfermedad de Alzheimer (CIE-10: G30)** mostró un **cambio en la tendencia** de incidencia post-pandemia. La Dra. Ruth advirtió que la caída en 2020 es atribuible a la concentración de servicios de salud en la emergencia sanitaria, y recomendó prudencia en la interpretación.

> *"La caída en reportes de Alzheimer durante 2020 no implica menor prevalencia sino menor detección — los servicios estaban enfocados en COVID-19."* — Dra. Ruth Pérez, IMSS (reunión 11-feb-2026)

In [ ]:
# =============================================================================
# 🟡 EJECUTAR BASELINE COMPLETO — ALZHEIMER (G30)
# =============================================================================
if DATOS.get("Alzheimer") is not None:
    RES_ALZ = ejecutar_padecimiento("Alzheimer", DATOS["Alzheimer"],
                                     PAL["Alzheimer"])
else:
    RES_ALZ = None
    logger.info("Alzheimer: datos no disponibles por separado")


---
## 9. 📊 Dashboard Ejecutivo Comparativo <a id='sec9'></a>

Dashboard consolidado que presenta el desempeño comparativo entre los **tres padecimientos** en todos los niveles de modelado. Diseñado para comunicación con stakeholders del IMSS.

In [ ]:
# =============================================================================
# COMPILAR RESULTADOS PARA EL DASHBOARD
# =============================================================================

def compilar_resumen(nombre, R, pal):
    """Genera filas de resumen para el dashboard."""
    if R is None:
        return []
    e = pal["emoji"]
    filas = []

    # Nacional
    r = R["nac"]["r"]
    filas.append({"Padecimiento": f"{e} {nombre}", "Nivel": "Nacional",
                  "RMSE_CV": r.get("rmse",np.nan), "MAE_CV": r.get("mae",np.nan),
                  "MAPE_CV(%)": r.get("mape",np.nan),
                  "MAPE_Train(%)": r.get("mape_train",np.nan),
                  "Outliers": r.get("n_outliers",0)})

    # Nacional Hombres / Mujeres
    for key, lbl in [("nac_h","Nacional ♂"), ("nac_m","Nacional ♀")]:
        r = R[key]["r"]
        filas.append({"Padecimiento": f"{e} {nombre}", "Nivel": lbl,
                      "RMSE_CV": r.get("rmse",np.nan), "MAE_CV": r.get("mae",np.nan),
                      "MAPE_CV(%)": r.get("mape",np.nan),
                      "MAPE_Train(%)": r.get("mape_train",np.nan),
                      "Outliers": r.get("n_outliers",0)})

    # Promedio Estados
    de = pd.DataFrame(R["edos"])
    filas.append({"Padecimiento": f"{e} {nombre}", "Nivel": "32 estados (prom)",
                  "RMSE_CV": de["rmse"].mean(), "MAE_CV": de["mae"].mean(),
                  "MAPE_CV(%)": de["mape"].mean(),
                  "MAPE_Train(%)": de["mape_train"].mean(),
                  "Outliers": int(de["n_outliers"].sum())})

    # Promedio Estado×Sexo
    des = pd.DataFrame(R["edo_sexo"])
    filas.append({"Padecimiento": f"{e} {nombre}", "Nivel": "64 edo×sexo (prom)",
                  "RMSE_CV": des["rmse"].mean(), "MAE_CV": des["mae"].mean(),
                  "MAPE_CV(%)": des["mape"].mean(),
                  "MAPE_Train(%)": des.get("mape_train", pd.Series([np.nan])).mean(),
                  "Outliers": "—"})
    return filas


filas = []
for n, R, p in [("Depresión",RES_DEP,PAL.get("Depresión",PAL.get("General"))),
                ("Parkinson",RES_PAR,PAL["Parkinson"]),
                ("Alzheimer",RES_ALZ,PAL["Alzheimer"])]:
    filas.extend(compilar_resumen(n, R, p))

df_dash = pd.DataFrame(filas).round(2)

print("\n" + "╔" + "═"*95 + "╗")
print("║" + "  📊 DASHBOARD EJECUTIVO — MODELO BASELINE EPIFORECAST-MX".center(95) + "║")
print("╚" + "═"*95 + "╝\n")
display(df_dash)


In [ ]:
# =============================================================================
# 📊 DASHBOARD VISUAL — 6 PANELES
# =============================================================================

# Datos para gráficos
pads_activos = []
for n, R, c in [("Depresión", RES_DEP, "#9B2242"),
                ("Parkinson", RES_PAR, "#00524E"),
                ("Alzheimer", RES_ALZ, "#B58500")]:
    if R is not None:
        pads_activos.append((n, R, c))

if len(pads_activos) == 0:
    print("⚠ Sin resultados suficientes para Dashboard.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(20, 11))
    fig.suptitle("DASHBOARD EJECUTIVO — Modelo Baseline EpiForecast-MX\n"
                 "Comparativa entre Padecimientos · Cross-Validation",
                 fontsize=16, fontweight="bold", y=1.02)
    fig.patch.set_facecolor("#FAFAFA")

    names = [n for n,_,_ in pads_activos]
    cols  = [c for _,_,c in pads_activos]
    mape_nac = [R["nac"]["r"]["mape"] for _,R,_ in pads_activos]
    rmse_nac = [R["nac"]["r"].get("rmse",0) for _,R,_ in pads_activos]
    mae_nac  = [R["nac"]["r"].get("mae",0) for _,R,_ in pads_activos]

    # ── Panel 1: MAPE Nacional ──────────────────────────────────────────
    ax = axes[0,0]
    bars = ax.bar(names, mape_nac, color=cols, edgecolor="white", lw=1.5)
    ax.axhline(30, color="#888", ls="--", alpha=0.5, label="Umbral aceptable (30%)")
    for b, v in zip(bars, mape_nac):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
                f"{v:.1f}%", ha="center", va="bottom", fontweight="bold")
    ax.set_ylabel("MAPE (%)"); ax.set_title("MAPE Nacional", fontweight="bold")
    ax.legend(fontsize=8)

    # ── Panel 2: RMSE Nacional ──────────────────────────────────────────
    ax = axes[0,1]
    bars = ax.bar(names, rmse_nac, color=cols, edgecolor="white", lw=1.5)
    for b, v in zip(bars, rmse_nac):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.2,
                f"{v:.1f}", ha="center", va="bottom", fontweight="bold")
    ax.set_ylabel("RMSE"); ax.set_title("RMSE Nacional", fontweight="bold")

    # ── Panel 3: MAE Nacional ───────────────────────────────────────────
    ax = axes[0,2]
    bars = ax.bar(names, mae_nac, color=cols, edgecolor="white", lw=1.5)
    for b, v in zip(bars, mae_nac):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.2,
                f"{v:.1f}", ha="center", va="bottom", fontweight="bold")
    ax.set_ylabel("MAE"); ax.set_title("MAE Nacional", fontweight="bold")

    # ── Panel 4: Train vs CV (sub/sobreajuste) ──────────────────────────
    ax = axes[1,0]
    x = np.arange(len(pads_activos)); w = 0.3
    t_vals = [R["nac"]["r"]["mape_train"] for _,R,_ in pads_activos]
    cv_vals = mape_nac
    ax.bar(x-w/2, t_vals, w, label="MAPE Train", color=cols, alpha=0.45, edgecolor="white")
    ax.bar(x+w/2, cv_vals, w, label="MAPE CV", color=cols, edgecolor="white")
    ax.set_xticks(x); ax.set_xticklabels(names)
    ax.set_ylabel("MAPE (%)"); ax.set_title("Sub/Sobreajuste (Train vs CV)", fontweight="bold")
    ax.legend(fontsize=8)

    # ── Panel 5: Distribución MAPE por estados (boxplot) ────────────────
    ax = axes[1,1]
    data_box, lbl_box, col_box = [], [], []
    for n, R, c in pads_activos:
        mps = pd.DataFrame(R["edos"])["mape"].dropna().values
        if len(mps) > 0:
            data_box.append(mps); lbl_box.append(n); col_box.append(c)
    if data_box:
        bp = ax.boxplot(data_box, labels=lbl_box, patch_artist=True,
                        widths=0.5, showfliers=True,
                        flierprops=dict(marker="^", markerfacecolor="red", ms=6))
        for p, c in zip(bp["boxes"], col_box):
            p.set_facecolor(c); p.set_alpha(0.55)
        for med in bp["medians"]:
            med.set_color("white"); med.set_linewidth(2)
    ax.axhline(30, color="#888", ls="--", alpha=0.5)
    ax.set_ylabel("MAPE (%)"); ax.set_title("Distribución MAPE Estatal", fontweight="bold")

    # ── Panel 6: Hombres vs Mujeres Nacional ────────────────────────────
    ax = axes[1,2]
    x = np.arange(len(pads_activos)); w = 0.3
    mh = [R["nac_h"]["r"]["mape"] for _,R,_ in pads_activos]
    mm = [R["nac_m"]["r"]["mape"] for _,R,_ in pads_activos]
    ax.bar(x-w/2, mh, w, label="♂ Hombres", color=IMSS["teal"], edgecolor="white")
    ax.bar(x+w/2, mm, w, label="♀ Mujeres", color=IMSS["burgundy"], edgecolor="white")
    ax.set_xticks(x); ax.set_xticklabels(names)
    ax.set_ylabel("MAPE (%)"); ax.set_title("MAPE Nacional ♂ vs ♀", fontweight="bold")
    ax.legend(fontsize=8)

    for a in axes.flat:
        a.grid(True, axis="y", ls="--", alpha=0.3)

    plt.tight_layout()
    plt.show(); plt.close(fig)


In [ ]:
# =============================================================================
# 📊 HEATMAP — MAPE POR ESTADO × PADECIMIENTO
# =============================================================================

heat_data = {}
for n, R, _ in pads_activos:
    de = pd.DataFrame(R["edos"])
    de["entidad"] = de["nombre"].str.split("—").str[-1].str.strip()
    heat_data[n] = de.set_index("entidad")["mape"]

if heat_data:
    df_heat = pd.DataFrame(heat_data).round(1)
    # Ordenar por MAPE promedio
    df_heat["prom"] = df_heat.mean(axis=1)
    df_heat = df_heat.sort_values("prom", ascending=True).drop(columns="prom")

    fig, ax = plt.subplots(figsize=(10, max(14, len(df_heat)*0.42)))
    fig.patch.set_facecolor("#FAFAFA")

    cmap = LinearSegmentedColormap.from_list("imss_heat",
        ["#00524E", "#4A9E9A", "#E8D5B5", "#D4758B", "#9B2242"])

    sns.heatmap(df_heat, annot=True, fmt=".0f", cmap=cmap,
                linewidths=0.5, linecolor="white",
                cbar_kws={"label": "MAPE (%)", "shrink": 0.6},
                ax=ax, vmin=0, vmax=100)

    ax.set_title("MAPE (%) por Entidad Federativa × Padecimiento\n"
                 "Modelo Baseline — Cross-Validation",
                 fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("Padecimiento", fontsize=12)
    ax.set_ylabel("")
    ax.tick_params(axis="y", labelsize=9)
    plt.tight_layout(); plt.show(); plt.close(fig)


---
## 10. Análisis de Sub/Sobreajuste <a id='sec10'></a>

Se comparan las métricas en **entrenamiento** vs. **validación cruzada** para detectar problemas de ajuste (Géron, 2022):

| Condición | Indicador | Acción en Avance 4 |
|---|---|---|
| **Subajuste** | Error alto en Train **Y** CV | Modelo demasiado simple → flexibilizar `changepoint_prior_scale` |
| **Buen ajuste** ✅ | Error bajo en Train, similar en CV | Modelo generaliza correctamente |
| **Sobreajuste** | Error bajo en Train, alto en CV | Modelo memoriza → incrementar regularización |

In [ ]:
# =============================================================================
# TABLA DE SUB/SOBREAJUSTE
# =============================================================================

filas_aj = []
for nombre, R, pal in [("🔴 Depresión", RES_DEP, PAL.get("Depresión")),
                        ("🟢 Parkinson", RES_PAR, PAL["Parkinson"]),
                        ("🟡 Alzheimer", RES_ALZ, PAL["Alzheimer"])]:
    if R is None:
        continue
    for nivel, key in [("Nacional","nac"), ("Nac. ♂","nac_h"), ("Nac. ♀","nac_m")]:
        r = R[key]["r"]
        mt = r.get("mape_train", 0)
        mc = r.get("mape", 0)
        gap = mc - mt
        if mt > 40:
            dx = "⚠️ Posible SUBAJUSTE"
        elif gap > 15:
            dx = "⚠️ Posible SOBREAJUSTE"
        else:
            dx = "✅ Ajuste aceptable"

        filas_aj.append({
            "Padecimiento": nombre, "Nivel": nivel,
            "MAPE Train (%)": round(mt, 2),
            "MAPE CV (%)": round(mc, 2),
            "Brecha (CV−Train)": round(gap, 2),
            "Diagnóstico": dx
        })

df_ajuste = pd.DataFrame(filas_aj)
print("\n" + "═"*100)
print("  ANÁLISIS DE SUB/SOBREAJUSTE — MODELOS NACIONALES POR PADECIMIENTO")
print("═"*100)
display(df_ajuste)


In [ ]:
# =============================================================================
# SCATTER PLOT: MAPE TRAIN vs CV POR ESTADO (TODOS LOS PADECIMIENTOS)
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor("#FAFAFA")

for n, R, c in pads_activos:
    de = pd.DataFrame(R["edos"])
    valid = de.dropna(subset=["mape","mape_train"])
    ax.scatter(valid["mape_train"], valid["mape"],
               s=40, color=c, alpha=0.6, edgecolors="white", lw=0.5,
               label=f"{n} ({len(valid)} estados)")

# Línea de igualdad (ajuste perfecto)
lim = ax.get_xlim()[1]
ax.plot([0, lim], [0, lim], "k--", alpha=0.3, label="Línea de ajuste perfecto")
ax.fill_between([0, lim], [0, lim], [15, lim+15],
                alpha=0.05, color="orange", label="Zona de sobreajuste (>15%)")

ax.set_xlabel("MAPE Train (%)", fontsize=12)
ax.set_ylabel("MAPE CV (%)", fontsize=12)
ax.set_title("Diagnóstico Sub/Sobreajuste por Estado y Padecimiento\n"
             "Puntos sobre la diagonal → mayor error en CV que en Train",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="upper left")
ax.grid(True, ls="--", alpha=0.3)
plt.tight_layout(); plt.show(); plt.close(fig)


---
## 11. Análisis de Componentes (Importancia de Características) <a id='sec11'></a>

En Prophet, la "importancia de características" se traduce en la **contribución relativa de los componentes** del modelo aditivo (Taylor & Letham, 2018):

$$\hat{y}(t) = g(t) + s(t) + \epsilon_t$$

Donde:
- $g(t)$ = **Tendencia**: evolución estructural a largo plazo
- $s(t)$ = **Estacionalidad anual**: patrones cíclicos de 52 semanas
- **Changepoints**: quiebres estructurales detectados automáticamente (e.g., COVID-19)

> A diferencia de modelos supervisados (XGBoost, RF) donde la importancia se asigna a *features* explícitas, Prophet opera sobre descomposición temporal. Las "características" relevantes son los componentes del modelo.

In [ ]:
# =============================================================================
# ANÁLISIS DE COMPONENTES POR PADECIMIENTO
# =============================================================================

for n, R, pal in pads_activos:
    fc = R["nac"]["forecast"]
    m  = R["nac"]["modelo"]
    tag = PAL[n]["tag"]

    comps = [c for c in ["trend","yearly"] if c in fc.columns]

    print(f"\n{'━'*65}")
    print(f"  📊 Componentes — {tag}")
    print(f"{'━'*65}")

    # Varianza explicada
    varianzas = {c: fc[c].var() for c in comps}
    total_var = sum(varianzas.values())
    for comp, var in varianzas.items():
        pct = (var/total_var)*100 if total_var > 0 else 0
        barra = "█" * int(pct/2) + "░" * (50 - int(pct/2))
        print(f"  {comp.capitalize():15s} {barra} {pct:.1f}%")

    # Changepoints
    cps_covid = [cp for cp in m.changepoints
                 if pd.Timestamp("2020-01-01") <= cp <= pd.Timestamp("2021-12-31")]
    print(f"\n  Changepoints totales: {len(m.changepoints)}")
    if cps_covid:
        print(f"  ✅ Changepoints durante COVID-19: {len(cps_covid)}")
        for cp in cps_covid:
            print(f"     → {cp.strftime('%Y-%m-%d')}")
    else:
        print(f"  ⚠ Sin changepoints explícitos durante COVID-19")


In [ ]:
# =============================================================================
# GRÁFICO COMPARATIVO DE VARIANZA POR COMPONENTE
# =============================================================================

fig, axes = plt.subplots(1, len(pads_activos), figsize=(5*len(pads_activos), 5))
if len(pads_activos) == 1:
    axes = [axes]

for ax, (n, R, c) in zip(axes, pads_activos):
    fc = R["nac"]["forecast"]
    comps = [co for co in ["trend","yearly"] if co in fc.columns]
    varianzas = {co: fc[co].var() for co in comps}
    total = sum(varianzas.values())
    pcts = {co: (v/total)*100 for co, v in varianzas.items()}

    bars = ax.barh(list(pcts.keys()), list(pcts.values()), color=c,
                   edgecolor="white", height=0.5)
    for b, v in zip(bars, pcts.values()):
        ax.text(b.get_width()+1, b.get_y()+b.get_height()/2,
                f"{v:.1f}%", va="center", fontweight="bold")
    ax.set_xlim(0, 110)
    ax.set_xlabel("% Varianza Explicada")
    ax.set_title(f"{PAL[n]['emoji']} {n}", fontweight="bold")
    ax.grid(True, axis="x", ls="--", alpha=0.3)

fig.suptitle("Contribución de Componentes por Padecimiento",
             fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show(); plt.close(fig)


---
## 12. Desempeño Mínimo Aceptable <a id='sec12'></a>

### 12.1 Umbrales Propuestos

No existe antecedente histórico de pronóstico epidemiológico automatizado para estos padecimientos en el IMSS. Los umbrales se basan en la literatura (Hyndman & Athanasopoulos, 2021) y los requisitos institucionales de la Dra. Ruth:

| Nivel | MAPE | Interpretación |
|---|---|---|
| 🟢 **Excelente** | ≤ 20% | Pronóstico confiable para decisiones operativas |
| 🟡 **Aceptable** | ≤ 30% | Útil para planificación estratégica |
| 🟠 **Requiere mejora** | ≤ 50% | Indicativo de viabilidad; requiere tuning en Avance 4 |
| 🔴 **Insuficiente** | > 50% | No viable sin mejora significativa |

In [ ]:
# =============================================================================
# VERIFICACIÓN DE DESEMPEÑO MÍNIMO POR PADECIMIENTO
# =============================================================================

print("\n" + "═"*85)
print("  VERIFICACIÓN DE DESEMPEÑO MÍNIMO ACEPTABLE")
print("═"*85)

total_modelos = 0

for n, R, pal in pads_activos:
    tag = PAL[n]["tag"]
    mape_nac = R["nac"]["r"]["mape"]
    de = pd.DataFrame(R["edos"])
    mapes = de["mape"].dropna()

    n_exc = (mapes <= 20).sum()
    n_acep = ((mapes > 20) & (mapes <= 30)).sum()
    n_mej = ((mapes > 30) & (mapes <= 50)).sum()
    n_ins = (mapes > 50).sum()

    estado_nac = "✅ CUMPLE" if mape_nac <= 30 else ("🟠 TUNING" if mape_nac <= 50 else "🔴 INSUFICIENTE")

    print(f"\n  {PAL[n]['emoji']} {tag}")
    print(f"  {'─'*55}")
    print(f"  Nacional MAPE: {mape_nac:.2f}%  {estado_nac}")
    print(f"  Distribución de {len(mapes)} estados:")
    print(f"    🟢 Excelente (≤20%):        {n_exc:2d} estados")
    print(f"    🟡 Aceptable (20-30%):      {n_acep:2d} estados")
    print(f"    🟠 Requiere mejora (30-50%): {n_mej:2d} estados")
    print(f"    🔴 Insuficiente (>50%):     {n_ins:2d} estados")

    cnt = 1 + 2 + len(R["edos"]) + len(R["edo_sexo"])
    total_modelos += cnt

print(f"\n{'═'*85}")
print(f"  🏥 TOTAL DE MODELOS BASELINE ENTRENADOS: {total_modelos}")
print(f"{'═'*85}")


---
## 13. Conclusiones y Siguientes Pasos <a id='sec13'></a>

### 13.1 Hallazgos Principales

1. **Viabilidad confirmada para los tres padecimientos**: Prophet captura la dinámica temporal de Depresión (F32), Parkinson (G20) y Alzheimer (G30). El desempeño supera consistentemente al modelo naïve.

2. **Cada padecimiento tiene un comportamiento epidemiológico distinto**: Se justifica la separación en modelos independientes, tal como recomendó la Dra. Grettel (reunión 12-feb: *"no es conveniente combinar los tres padecimientos porque tienen dinámicas epidemiológicas diferentes"*).

3. **Efecto COVID-19 capturado**: Los *changepoints* detectados coinciden con el período de emergencia sanitaria (mar 2020 – jun 2021), validando la robustez de Prophet ante quiebres estructurales.

4. **Variabilidad significativa entre estados**: La dispersión del MAPE por entidad federativa valida la decisión de entrenar modelos individuales por estado. Estados con bajo reporte presentan mayor incertidumbre (advertencia de la Dra. Grettel sobre estados con "muy pocos datos").

5. **Outliers identificados y documentados**: La detección de valores atípicos por IQR permite a los stakeholders del IMSS comprender la variabilidad inherente en los datos epidemiológicos.

6. **Diferencias por sexo**: Los modelos desagregados evidencian niveles de predecibilidad distintos entre hombres y mujeres, relevante para el prorrateo proporcional solicitado por las doctoras del IMSS.

### 13.2 Del Baseline al Modelo Tuneado (Avance 4)

| Hiperparámetro | Valor Baseline | Exploración en Avance 4 |
|---|---|---|
| `seasonality_mode` | `additive` | `multiplicative` para estados con estacionalidad proporcional |
| `changepoint_prior_scale` | 0.05 | Grid: [0.01, 0.05, 0.1, 0.5, 1.0] |
| `seasonality_prior_scale` | 10.0 | Grid: [0.1, 0.5, 1.0, 5.0, 10.0] |
| `holidays` | ninguno | Días festivos mexicanos y períodos vacacionales |
| `regressors` | ninguno | Variables exógenas (densidad poblacional, región socioeconómica) |

### 13.3 Alineación con Objetivos Estratégicos

- **Académico**: Cumple los objetivos 3.1 y 3.2 del Módulo 3 (TC5035) — métricas de calidad + modelo de referencia.
- **Científico**: Línea base reproducible para los dos artículos en desarrollo y los congresos de Estocolmo y Portugal.
- **Operativo**: Pronósticos baseline ya potencialmente útiles para planificación inicial en IMSS; modelos tuneados mejorarán precisión.

---
## 14. Referencias <a id='sec14'></a>

- Costa, R. (2022). *The CRISP-ML Methodology: A Step-by-Step Approach to Real-World Machine Learning Projects*. Edición propia.

- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3.ª ed.). O'Reilly Media.

- Hyndman, R. J., y Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3.ª ed.). OTexts. https://otexts.com/fpp3/

- Hyndman, R. J., y Koehler, A. B. (2006). Another look at measures of forecast accuracy. *International Journal of Forecasting*, 22(4), 679–688. https://doi.org/10.1016/j.ijforecast.2006.03.001

- Piccini, N. (2023, julio 19). 101 machine learning algorithms for data science with cheat sheets. *Data Science Dojo*. https://datasciencedojo.com/blog/machine-learning-algorithms/

- Studer, S., Bui, T. B., Drescher, C., Hanuschkin, A., Winkler, L., Peters, S., y Müller, K.-R. (2021). Towards CRISP-ML(Q): A Machine Learning Process Model with Quality Assurance Methodology. *Preprints*, 2021, 1, 0. https://doi.org/10.48550/arXiv.2003.05155

- Taylor, S. J., y Letham, B. (2018). Forecasting at scale. *The American Statistician*, 72(1), 37–45. https://doi.org/10.1080/00031305.2017.1380080

- Visengeriyeva, L., Kammer, A., Bär, I., Kniesz, A., y Plöd, M. (2023). CRISP-ML(Q). *The ML Lifecycle Process. MLOps. INNOQ*. https://ml-ops.org/content/crisp-ml

---

<center>

**EpiForecast-MX** · Proyecto en colaboración con el Instituto Mexicano del Seguro Social (IMSS)

Tecnológico de Monterrey · Maestría en Inteligencia Artificial Aplicada · 2026

</center>